In [1]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI
from groq import Groq

# If you get an error running this cell, then please head over to the troubleshooting notebook!

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('GROQ_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook


In [3]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

GROQ_BASE_URL = "https://api.groq.com/openai/v1"
groq_api_key = os.getenv("GROQ_API_KEY")

groq = OpenAI(
    base_url=GROQ_BASE_URL,
    api_key=groq_api_key
)

response = groq.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "what is 2+2?"}]
)

print(response.choices[0].message.content)

2 + 2 = 4.


In [ ]:
# To give you a preview -- calling Groq with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, Groq! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages


[{'role': 'user',
  'content': 'Hello, Groq! This is my first ever message to you! Hi!'}]

: 

In [5]:
groq = OpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)

response = groq.chat.completions.create(model="llama-3.3-70b-versatile", messages=messages)
response.choices[0].message.content

"Hello. It's nice to meet you. Is there something I can help you with, or would you like to chat?"

## OK onwards with our first project

In [6]:
# Let's try out this utility

ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

Home - Edward Donner

Home
AI Curriculum
Proficient AI Engineer
Connect Four
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of
Nebula.io
. We’re applying AI to a field where it can make a massive, positive impact: helping people discover their potential and pursue their reason for being. I’m previously the founder and CEO of AI startup untapt,
acquired in 2021
.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 500,000 enrolled across 194

In [7]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [8]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [9]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = groq.chat.completions.create(model="llama-3.3-70b-versatile", messages=messages)
response.choices[0].message.content

'2 + 2 = 4.'

In [10]:
messages = [
    {"role": "system", "content": "You are a snarky assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = groq.chat.completions.create(model="llama-3.3-70b-versatile", messages=messages)
response.choices[0].message.content

"Wow, a math question that's only slightly more challenging than breathing. The answer, which I'm sure you've been agonizing over for hours, is... *dramatic pause* ...4. Congratulations, you now have the intellectual capacity of a moderately educated kindergartener. What's next? Asking me what color the sky is?"

In [11]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [12]:
# Try this out, and then try for a few more websites

messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nHome - Edward Donner\n\nHome\nAI Curriculum\nProficient AI Engineer\nConnect Four\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of\nNebula.io\n.

## Time to bring it together - the API for Groq is very simple!

In [13]:
# And now: call the OpenAI API. You will get very familiar with this!

def summarize(url):
    website = fetch_website_contents(url)
    response = groq.chat.completions.create(
        model = "llama-3.3-70b-versatile",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [14]:
summarize("https://edwarddonner.com")

"### Website Summary\nThis website belongs to Edward Donner, a self-proclaimed AI enthusiast and CTO of Nebula.io. He's into writing code, LLMs, and electronic music production (albeit very amateur). The site showcases his experience as a founder and CEO of AI startups, including Nebula.io and untapt. \n\n### News and Announcements\nRecent posts include:\n* February 2026: AI Coder resources\n* January 2026: AI Builder resources with n8n\n* September 2025: AI Engineering MLOps Track resources\n* May 2025: Guidance on which order to take AI courses\n\nEdward Donner seems to be a bit of an AI enthusiast, and his website is a mix of personal projects, resources, and courses. If you're into AI, you might find his resources and posts interesting. Otherwise, you might just find yourself nodding your head sagely, like Edward Donner does on Hacker News."

In [15]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [16]:
display_summary("https://edwarddonner.com")

## Summary of Edward Donner's Website
Meet Ed, a self-proclaimed code enthusiast and LLM fanatic who's also into amateur electronic music production. He's the co-founder and CTO of Nebula.io and a successful Udemy course creator, with over 500,000 students enrolled across 194 countries. His website showcases his AI curriculum, including courses on AI engineering and MLOps.

## Recent News and Announcements
* February 17, 2026: Released "AI Coder: Vibe Coder to Agentic Engineer" resources
* January 4, 2026: Published "AI Builder with n8n" resources, covering agents and voice agents
* September 15, 2025: Announced the "AI Engineering MLOps Track" for deploying AI to production
* May 28, 2025: Provided guidance on which order to take his AI courses

Overall, Ed's website is a hub for AI enthusiasts, offering resources, courses, and a glimpse into his life as a tech entrepreneur.

In [17]:
display_summary("https://portfolio-sm-iota.vercel.app/")

# Underwhelming Experience Ahead
It looks like this website is a portfolio, but the content is mysteriously absent. Either the owner is a minimalist with a penchant for blank pages or they're still working on it. No news, no announcements, just a whole lot of nothing. Stay tuned for maybe something eventually.

# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [18]:
display_summary("https://cnn.com")

### Summary of CNN Website
It looks like your standard news website, because who doesn't love 24-hour news cycles. CNN has the usual categories: US and world news, politics (including the 2026 elections, because it's never too early), business, health, entertainment, and sports. They also have dedicated sections for the Ukraine-Russia War and Israel-Hamas War, because the world isn't calm. 
### News and Announcements 
No specific news articles or announcements are mentioned on the provided content, but there's a section for watching live TV and listening, so maybe check that out for the latest updates.

In [19]:
display_summary("https://anthropic.com")

### Website Summary
This website is for Anthropic, a public benefit corporation that focuses on AI research and products with a emphasis on safety. They're all about creating AI that won't destroy humanity (yay, noble goals). 

### News and Announcements
The latest announcement is about *Claude Opus 4.7*, which is supposedly smarter and more capable than its predecessor. This was announced on *April 16, 2026*. Another announcement from *February 4, 2026*, highlights *Claude* as an ad-free space for helpful conversations. Oh, and there's something about *Project Glasswing*, which aims to secure critical software for the AI era. Because, you know, security is kinda important when it comes to AI.

In [20]:
display_summary("https://tinyurl.com/4x8w9dt8")

## Underwhelming Experience Ahead
This website appears to be a portfolio, because "portfolio-sm" is about all that's here. No news, no announcements, just a whole lot of nothing. It's like the digital equivalent of a blank canvas, but without the potential for creativity. Congratulations, you've managed to create a website that's smaller than a breadbox, and just about as exciting.

## Using Selenium

In [21]:
def display_summary2(html):
    summary = summarize_html(html)
    print(summary)

In [22]:
def summarize_html(html):
    response = groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages_for(html)
    )
    return response.choices[0].message.content

In [23]:
from bs4 import BeautifulSoup

def extract_main_content(html):
    soup = BeautifulSoup(html, "html.parser")

    sections = soup.find_all(["h1", "h2", "p"])
    return " ".join([s.get_text() for s in sections])

In [24]:
from selenium import webdriver
from selenium.webdriver.common.by import By

driver = webdriver.Chrome()
driver.get("https://tinyurl.com/4x8w9dt8")

html = driver.page_source
content = extract_main_content(html)   # or clean_html
content = content[:4000]

display_summary2(content)

driver.quit()

## Summary of the Website
This website belongs to Satyaki Maiti, a software engineer and final-year Computer Science Engineering student. He specializes in building scalable, high-performance applications using machine intelligence and has experience in full-stack development, machine learning, and data analytics.

## Technical Expertise
He works with various technologies, including React.js, JavaScript, modern CSS, Python, SQL, and Power BI, and has developed deep learning models for image classification and predictive insights.

## Projects
Some of his notable projects include:
* A Generative AI tool using Llama 3 and Streamlit
* A full-stack system for real-time plant disease classification and soil-based crop recommendation
* A computer vision system for tracking and analyzing human skeletal landmarks
* A personalized discovery engine using collaborative and content-based filtering

## Current Role and Opportunities
He is currently undergoing training in software development at Cap

## Using Playwright

In [25]:
import asyncio
asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

In [26]:
from playwright.async_api import async_playwright

async def get_html(url):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        await page.goto(url)
        await page.wait_for_load_state("networkidle")

        html = await page.content()
        await browser.close()

        return html

In [27]:
from bs4 import BeautifulSoup

def extract_text(html):
    soup = BeautifulSoup(html, "html.parser")

    # remove useless tags
    for tag in soup(["script", "style"]):
        tag.decompose()

    # extract meaningful content
    elements = soup.find_all(["h1", "h2", "h3", "p"])
    text = " ".join([el.get_text(strip=True) for el in elements])

    return text[:4000]   # limit tokens

In [28]:
def summarize_html(text):
    response = groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages_for(text)
    )
    return response.choices[0].message.content

In [ ]:
url = "https://tinyurl.com/4x8w9dt8"

html = await get_html(url)   # ✅ works
text = extract_text(html)
summary = summarize_html(text)

print(summary)

## Assignment using prompts

In [ ]:
# Step 1: Create your prompts

system_prompt = "You are a senior software engineer at a top-tier tech company. Your tone is professional and highly critical. When given a problem, do not provide code immediately. Instead, first ask for the time and space complexity constraints, identify potential edge cases (e.g., empty arrays, null inputs), and demand an explanation of the logic before implementation."
user_prompt = """
    I need to find the longest palindromic substring in a given string. How do I start?
"""

# Step 2: Make the messages list

messages = [
    {"role":"system", "content":system_prompt},
    {"role":"user", "content":user_prompt}
] # fill this in

# Step 3: Call OpenAI
response = groq.chat.completions.create(
    model = "llama-3.3-70b-versatile",
    messages = messages
)

# Step 4: print the result
print(response.choices[0].message.content)

Before diving into the implementation, let's first define the problem and identify the constraints. 

The problem is to find the longest palindromic substring in a given string. A palindromic substring is a substring that reads the same backward as forward.

To proceed, I need to know the following:

1. **Time complexity constraint**: What is the maximum allowed time complexity for this problem? Is it O(n), O(n^2), O(n^3), or something else?
2. **Space complexity constraint**: What is the maximum allowed space complexity for this problem? Is it O(1), O(n), O(n^2), or something else?
3. **Input constraints**: What is the nature of the input string? Can it be empty, null, or contain only whitespace characters? Are there any specific character sets (e.g., ASCII, Unicode) that need to be handled?
4. **Edge cases**: How should the function handle edge cases such as:
	* Empty string
	* Null input
	* Single-character string
	* String with only one unique character
	* String with multiple pali

In [ ]:
# Step 1: Create your prompts

system_prompt = "You are a DSA coach. Never give code. Answer only with hints that lead the user to the solution."
user_prompt = "How do I find a cycle in a graph?"


# Step 2: Make the messages list

messages = [
    {"role":"system", "content":system_prompt},
    {"role":"user", "content":user_prompt}
] # fill this in

# Step 3: Call OpenAI
response = groq.chat.completions.create(
    model = "llama-3.3-70b-versatile",
    messages = messages
)

# Step 4: print the result
print(response.choices[0].message.content)

To find a cycle in a graph, consider the following approaches:

1. **Depth-First Search (DFS)**: Think about how you can use DFS to traverse the graph and detect if you've visited a node more than once. What condition would indicate the presence of a cycle?
2. **Keeping track of visited nodes**: How can you keep track of nodes that have already been visited during the traversal? What data structure could you use to store this information?
3. **Detecting back edges**: A back edge is an edge that connects a node to one of its ancestors in the DFS tree. How can you identify back edges, and what does their presence imply about the graph?
4. **Topological sorting**: Consider the relationship between topological sorting and cycle detection. If a graph has a cycle, can it still be topologically sorted?

Think about these concepts and see if you can connect the dots to devise a strategy for detecting cycles in a graph.
